# Tool calling — end-to-end loop

A complete agentic loop with the Open Responses API:

1. Define tools with JSON-schema parameters.
2. Ask something that requires a tool.
3. The model returns `function_call` items (not the final answer).
4. **You** execute the tool and feed the result back as a `function_call_output` item.
5. The model produces the final answer with the tool result in context.

This is the pattern behind every agent: *model proposes → code executes → model synthesizes*.

In [ ]:
import json
from aura import AuraClient, Tool

client = AuraClient()

In [ ]:
# 1. Define a toy weather tool (JSON-schema parameters)
weather_tool = Tool.function_tool(
    name="get_weather",
    description="Get the current weather for a location",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. Tokyo",
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
            },
        },
        "required": ["location"],
    },
)

In [ ]:
# 2. Simulated tool executor (swap for a real API call)
def run_tool(name: str, arguments: str) -> str:
    args = json.loads(arguments)
    if name == "get_weather":
        unit = args.get("unit", "celsius")
        temp = 22 if unit == "celsius" else 72
        return json.dumps({
            "location": args["location"],
            "temperature": temp,
            "unit": unit,
            "conditions": "sunny",
        })
    return json.dumps({"error": f"unknown tool {name}"})

In [ ]:
# 3. Ask something that needs the tool
response = client.responses.create(
    model="gpt-5.4-mini",
    input="What's the weather in Tokyo? Reply with the temperature and conditions.",
    tools=[weather_tool],
)

print(f"status: {response.status}")
print(f"tool calls: {len(response.tool_calls)}")
for tc in response.tool_calls:
    print(f"  -> {tc.name}({tc.arguments})")

In [ ]:
# 4. Execute + feed results back (the complete loop)

# The Responses API expects function_call / function_call_output
# items as plain dicts on the wire (see the Open Responses spec).
messages = []
response = client.responses.create(
    model="gpt-5.4-mini",
    input="What's the weather in Tokyo and in Paris?",
    tools=[weather_tool],
)

while response.has_tool_calls:
    # append the model's function_call items so the next turn sees them
    for tc in response.tool_calls:
        messages.append({
            "type": "function_call",
            "call_id": tc.call_id,
            "name": tc.name,
            "arguments": tc.arguments,
        })
        result = run_tool(tc.name, tc.arguments)
        print(f"tool {tc.name} -> {result}")
        messages.append({
            "type": "function_call_output",
            "call_id": tc.call_id,
            "output": result,
        })

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[weather_tool],
        previous_response_id=response.id,
    )

print(f"\nfinal: {response.output_text}")

The `while` loop keeps running as long as the model emits tool calls — that's how multi-step agents chain several tool invocations in one conversation.